# 第 5 章实践：GEMM+Bias+GELU


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）混合 Kernel 中，一个逻辑 Block = 1 AIC + ______ 个 AIV；AIV 侧的全局核索引换算 AI Core 编号要除以 ______。
2. （选择）以下哪种中间数据更适合走 GM workspace 而非片上通路？
   A. 一次消费的 [64,64] FP32 Tile<br/>
   B. 需要三遍读取的完整行数据
3. （判断）`auto_mutex=True` 可以管理 AIC 与 AIV 之间的数据依赖。（对/错）

In [ ]:
!cat ./answer/05.05_objective_answer.txt

## 2. 编程实践一（简单）：GEMM + Bias（Vector 后处理）

**任务**：实现 `out = A@B + bias`（[256,128]×[128,256] FP16 → FP32，bias 为 [256] 广播到每行）。Cube 完成矩阵乘写入 workspace，Vector 读出加 bias 写回。

**提示**：bias 按 [1,TILE_N] 行 Tile 加载。本章先用 `pl.neg(bias_tile,bias_tile)` 在 UB 内原地取负，再用 `pl.expand_sub(out_tile,mat_tile,bias_tile,dim=1)` 实现加 Bias；GM 中的 bias 不会被修改。`pl.expand_add` 属于版本相关能力，不能假定目标安装包提供该接口。事件：Cube FIX 发、Vector MTE2 等。


In [ ]:
# 在此完成实践一
pass

## 3. 编程实践二（中等）：GEMM+Bias+GELU

**任务**：在实践一基础上接 GELU 激活（`gelu(x) = 0.5x(1+tanh(0.7978845608(x+0.044715x³)))`，这里采用 tanh 近似）。当前 Tile/VF API 没有直接的 gelu/tanh；可用 abs/exp/四则运算及比较选择构造稳定的 tanh（见答案），不能写未提供的 pl.tanh。Golden 必须用 `torch.nn.functional.gelu(..., approximate="tanh")`。


In [ ]:
# 在此完成实践二
pass

## 4. 编程实践三（困难）：流水化改造与耗时对比

**任务**：保持实践二功能不变，做两件事：

1. 把 Kernel 改成"Cube 沿 N 块循环产出、Vector 按任务错位消费"的流水版（两槽缓冲，task_id % 2 选槽）；
2. 记录（950 环境中）串行版与流水版的耗时对比，填入下方表格。

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">版本</th><th style="text-align:left;">耗时（ms）</th><th style="text-align:left;">加速比</th></tr>
<tr><td>串行（单槽、逐任务事件）</td><td>【950 待测】</td><td>1.0x</td></tr>
<tr><td>流水（两槽错位）</td><td>【950 待测】</td><td>【待测】</td></tr>
</table>


In [ ]:
!cat ./answer/05.05_practice_answer.txt

### 完整参考实现与验收

参考源码包含 `gemm_bias`、`gemm_gelu_serial`、`gemm_gelu_double` 与稳定 GELU VF。三者使用同一输入规格；先通过 Bias 中间结果，再验证激活，最后比较单槽/双槽。GM workspace 方案用于核对协议，不代表片上融合的最终性能。

[参考源码](./src/fused_gemm.py) 可在 950 上从章节目录执行 `python src/fused_gemm.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/fused_gemm.py


## 5. 本节小结

GEMM+Bias+GELU 走通了"后处理型融合"的完整路径：**任务映射 → Cube 产出 → 跨核同步 → Vector 后处理**，并体验了流水化改造。第 6~7 章转向 SIMT 轨道；第 9 章将把本章流水方法与 SIMT 结合，第 10 章 FlashAttention 是融合算子的集大成者。